In [1]:
# ============================================================
# PS S6E5 - exp_20260520_041_xgb_shared001v2_features_min
#
# Base:
#   016: exp_20260508_016_xgb_shared005_fe_te_seedens_min
#
# Change from 016:
#   Add the core feature representation used by the strong RealMLP 029/038 family:
#   - keep raw PitStop
#   - PitStop_cat_ and _PitStop_cat_count
#   - floored numerical category features
#   - RaceProgress_200_quantile_bin_
#   - LapTime (s)_7_quantile_bin_
#   - Race_Compound_ and Race_Year_
#   - count encoding for the added categorical/bin features
#
# Purpose:
#   Test whether the strong 029/038 RealMLP feature representation can improve
#   the XGB branch 016/017.
#
# Keep 016 implementation as-is as much as possible:
#   - XGBClassifier
#   - original rows appended to fold train
#   - Frequency Encoding
#   - cuML TargetEncoder
#   - 5 outer folds
#   - 5 seeds per fold
#   - save OOF/pred npy for blend
#
# Guardrails:
#   - Do not add 039 LOG features in this first XGB transfer test.
#   - Normalized_TyreLife is explicitly dropped from original.
#   - Do not use Normalized_TyreLife or proxy reconstruction.
#   - No pseudo labels.
#   - No endpoint/future proxy.
#
# Role:
#   XGB branch update candidate, not a single-model main candidate.
# ============================================================


In [2]:
import os
import gc
import json
import random
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.max_columns", 300)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss

import cudf
from cuml.preprocessing import TargetEncoder as cuTargetEncoder

from xgboost import XGBClassifier

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260520_041_xgb_shared001v2_features_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    TARGET_CANDIDATES = ["PitNextLap", "PitNextLab"]
    TARGET = "PitNextLap"
    ID_COL = "id"
    DANGER_COL = "Normalized_TyreLife"

    COMP_PATHS = [
        "/kaggle/input/competitions/playground-series-s6e5",
        "/kaggle/input/playground-series-s6e5",
        ".",
    ]

    ORIGINAL_PATHS = [
        "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
        "/kaggle/input/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
        "./f1_strategy_dataset_v4.csv",
    ]

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    SEED = 42
    N_SPLITS = 5

    N_TE_FOLDS = 5
    TE_SMOOTH = 20

    SEEDS = [42, 43, 44, 45, 46]

    USE_ORIGINAL = True

    LOSSGUIDE_MAX_LEAVES = 64

    XGB_PARAMS = {
        "n_estimators": 10000,
        "learning_rate": 0.03,

        "grow_policy": "lossguide",
        "max_depth": 0,
        "max_leaves": LOSSGUIDE_MAX_LEAVES,

        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,

        "reg_alpha": 0.0,
        "reg_lambda": 2.0,

        "objective": "binary:logistic",
        "eval_metric": "auc",

        "tree_method": "hist",
        "device": "cuda",
        "n_jobs": -1,
        "enable_categorical": True,

        "early_stopping_rounds": 200,
    }

    # 041: add RealMLP 029/038-style feature representation to XGB.
    USE_SHARED001V2_FEATURES_041 = True
    KEEP_RAW_PITSTOP_041 = True
    ADD_RACE_COMPOUND_YEAR_CROSSES_041 = True
    ADD_039_LOG_FEATURES = False

    CLIP_LOW = 1e-6
    CLIP_HIGH = 1.0 - 1e-6

    VERBOSE = 200


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def print_section(title: str) -> None:
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def first_existing_dir(paths):
    for p in paths:
        path = Path(p)
        if (path / "train.csv").exists() and (path / "test.csv").exists():
            return path
    raise FileNotFoundError(f"No valid competition path found: {paths}")


def first_existing_file(paths, required=True):
    for p in paths:
        path = Path(p)
        if path.exists():
            return path
    if required:
        raise FileNotFoundError(f"No valid file path found: {paths}")
    return None


def resolve_target_column(df: pd.DataFrame) -> str:
    for c in CFG.TARGET_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"No target column found among {CFG.TARGET_CANDIDATES}")


def clip_pred(x):
    return np.clip(x, CFG.CLIP_LOW, CFG.CLIP_HIGH)


def to_numpy(x):
    if hasattr(x, "get"):
        return x.get()
    return np.asarray(x)


def safe_colname(c):
    return (
        c.replace(" ", "_")
         .replace("(", "")
         .replace(")", "")
         .replace("/", "_")
         .replace("-", "_")
    )


def to_numeric_array(s):
    x = pd.to_numeric(s, errors="coerce").astype(float).values
    x = np.round(x, 6)
    return x


def round_to_step(s, step):
    return np.round(s / step) * step


seed_everything(CFG.SEED)

In [5]:
# ============================================================
# Load Data
# ============================================================

print_section("Load Data")

COMP_PATH = first_existing_dir(CFG.COMP_PATHS)

train = pd.read_csv(COMP_PATH / "train.csv")
test = pd.read_csv(COMP_PATH / "test.csv")
sample_submission = pd.read_csv(COMP_PATH / "sample_submission.csv")

target_col_train = resolve_target_column(train)
if target_col_train != CFG.TARGET:
    train = train.rename(columns={target_col_train: CFG.TARGET})

target_col_sub = resolve_target_column(sample_submission)
if target_col_sub != CFG.TARGET:
    sample_submission = sample_submission.rename(columns={target_col_sub: CFG.TARGET})

original_path = first_existing_file(CFG.ORIGINAL_PATHS, required=False)
orig = None

if CFG.USE_ORIGINAL and original_path is not None:
    orig = pd.read_csv(original_path)

    if CFG.TARGET not in orig.columns:
        target_col_orig = resolve_target_column(orig)
        orig = orig.rename(columns={target_col_orig: CFG.TARGET})

    if CFG.DANGER_COL in orig.columns:
        orig = orig.drop(columns=[CFG.DANGER_COL])
        print(f"Dropped danger column from original: {CFG.DANGER_COL}")

print("COMP_PATH:", COMP_PATH)
print("train:", train.shape)
print("test :", test.shape)
print("sample_submission:", sample_submission.shape)

if orig is not None:
    print("orig:", orig.shape)
    print("original path:", original_path)

print("target mean train:", train[CFG.TARGET].mean())
if orig is not None:
    print("target mean orig :", orig[CFG.TARGET].mean())

assert CFG.ID_COL in train.columns
assert CFG.ID_COL in test.columns
assert CFG.ID_COL in sample_submission.columns
assert CFG.TARGET in train.columns
assert CFG.TARGET in sample_submission.columns
assert test[CFG.ID_COL].equals(sample_submission[CFG.ID_COL])

if orig is not None:
    assert CFG.TARGET in orig.columns
    assert CFG.DANGER_COL not in orig.columns

train_ids = train[CFG.ID_COL].copy()
test_ids = test[CFG.ID_COL].copy()


Load Data
Dropped danger column from original: Normalized_TyreLife
COMP_PATH: /kaggle/input/competitions/playground-series-s6e5
train: (439140, 16)
test : (188165, 15)
sample_submission: (188165, 2)
orig: (101371, 15)
original path: /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv
target mean train: 0.19898210137996994
target mean orig : 0.25479673673930414


In [6]:
# ============================================================
# Base Columns
# ============================================================

TARGET = CFG.TARGET

BASE = [col for col in train.columns if col not in [CFG.ID_COL, TARGET]]
CATS = [col for col in BASE if train[col].dtype == "object"]

print(len(BASE), "Baseline Features.")
print(len(CATS), "Categorical Features.")
print("Categorical Columns:", CATS)

# shared_005 encoding:
# Driver / Compound / Race are mapped using train + test + original combined values.
# This is not target leakage, but it does use category vocabulary from test/orig.
for col in CATS:
    frames = [
        train[col].astype(str),
        test[col].astype(str),
    ]
    if orig is not None and col in orig.columns:
        frames.append(orig[col].astype(str))

    combined = pd.concat(frames, axis=0)
    uniques = combined.unique()
    mapping = {v: i for i, v in enumerate(uniques)}

    train[col] = train[col].astype(str).map(mapping).astype("int32").astype("category")
    test[col] = test[col].astype(str).map(mapping).astype("int32").astype("category")

    if orig is not None and col in orig.columns:
        orig[col] = orig[col].astype(str).map(mapping).astype("int32").astype("category")

14 Baseline Features.
3 Categorical Features.
Categorical Columns: ['Driver', 'Compound', 'Race']


In [7]:
# ============================================================
# Feature Engineering: NUM as CAT
# ============================================================

print_section("Feature Engineering: NUM as CAT")

NUM_as_CAT = []

NUM_CAT_BASE = [
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
]

for c in NUM_CAT_BASE:
    new_col = f"{c}_cat"
    for df in [train, test] + ([orig] if orig is not None else []):
        if c in df.columns:
            df[new_col] = df[c].astype(str)
    NUM_as_CAT.append(new_col)

ROUND_CONFIG = {
    "LapTime (s)": {
        "round_digits": [1, 0],
        "round_steps": [0.5, 1.0, 2.0, 5.0],
    },
    "LapTime_Delta": {
        "round_digits": [1, 0],
        "round_steps": [0.5, 1.0, 2.0, 5.0, 10.0],
    },
    "Cumulative_Degradation": {
        "round_digits": [1, 0],
        "round_steps": [1.0, 2.0, 5.0, 10.0, 20.0],
    },
}

for c, cfg in ROUND_CONFIG.items():
    for d in cfg["round_digits"]:
        new_col = f"{c}_round{d}_cat"
        for df in [train, test] + ([orig] if orig is not None else []):
            if c in df.columns:
                df[new_col] = df[c].round(d).astype(str)
        NUM_as_CAT.append(new_col)

    for step in cfg["round_steps"]:
        step_name = str(step).replace(".", "p")
        new_col = f"{c}_round_step_{step_name}_cat"

        for df in [train, test] + ([orig] if orig is not None else []):
            if c in df.columns:
                df[new_col] = round_to_step(df[c], step).astype(str)

        NUM_as_CAT.append(new_col)

NUM_as_CAT = [c for c in NUM_as_CAT if c in train.columns and c in test.columns]
if orig is not None:
    NUM_as_CAT = [c for c in NUM_as_CAT if c in orig.columns]

print(len(NUM_as_CAT), "Total Num->Cat Features Created!")
print(NUM_as_CAT)


Feature Engineering: NUM as CAT
23 Total Num->Cat Features Created!
['LapTime (s)_cat', 'LapTime_Delta_cat', 'Cumulative_Degradation_cat', 'LapTime (s)_round1_cat', 'LapTime (s)_round0_cat', 'LapTime (s)_round_step_0p5_cat', 'LapTime (s)_round_step_1p0_cat', 'LapTime (s)_round_step_2p0_cat', 'LapTime (s)_round_step_5p0_cat', 'LapTime_Delta_round1_cat', 'LapTime_Delta_round0_cat', 'LapTime_Delta_round_step_0p5_cat', 'LapTime_Delta_round_step_1p0_cat', 'LapTime_Delta_round_step_2p0_cat', 'LapTime_Delta_round_step_5p0_cat', 'LapTime_Delta_round_step_10p0_cat', 'Cumulative_Degradation_round1_cat', 'Cumulative_Degradation_round0_cat', 'Cumulative_Degradation_round_step_1p0_cat', 'Cumulative_Degradation_round_step_2p0_cat', 'Cumulative_Degradation_round_step_5p0_cat', 'Cumulative_Degradation_round_step_10p0_cat', 'Cumulative_Degradation_round_step_20p0_cat']


In [8]:
# ============================================================
# Feature Engineering: DIGIT
# ============================================================

print_section("Feature Engineering: DIGIT")

DIGIT_FEATURES = []

DIGIT_BASE = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
]

DECIMAL_DIGIT_BASE = [
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
]

INT_POSITIONS = [1, 10, 100, 1000]
DECIMAL_POSITIONS = [1, 2, 3]

all_dfs = [train, test] + ([orig] if orig is not None else [])

for c in DIGIT_BASE:
    if not all(c in df.columns for df in all_dfs):
        print(f"[Skip] {c} is not found in all dataframes.")
        continue

    sc = safe_colname(c)

    sign_col = f"{sc}_sign"
    for df in all_dfs:
        x = to_numeric_array(df[c])
        sign = np.sign(np.nan_to_num(x, nan=0.0)).astype(np.int8)
        df[sign_col] = sign
    DIGIT_FEATURES.append(sign_col)

    for p in INT_POSITIONS:
        nc = f"{sc}_digit_{p}s"

        for df in all_dfs:
            x = to_numeric_array(df[c])
            x_abs = np.abs(np.nan_to_num(x, nan=0.0))
            int_part = np.floor(x_abs).astype(np.int64)
            digit = ((int_part // p) % 10).astype(np.int8)
            df[nc] = digit

        DIGIT_FEATURES.append(nc)

for c in DECIMAL_DIGIT_BASE:
    if not all(c in df.columns for df in all_dfs):
        print(f"[Skip] {c} is not found in all dataframes.")
        continue

    sc = safe_colname(c)

    for d in DECIMAL_POSITIONS:
        nc = f"{sc}_decimal_digit_{d}"

        for df in all_dfs:
            x = to_numeric_array(df[c])
            x_abs = np.abs(np.nan_to_num(x, nan=0.0))
            digit = (np.floor(x_abs * (10 ** d)).astype(np.int64) % 10).astype(np.int8)
            df[nc] = digit

        DIGIT_FEATURES.append(nc)

DIGIT_FEATURES = [c for c in DIGIT_FEATURES if c in train.columns and c in test.columns]
if orig is not None:
    DIGIT_FEATURES = [c for c in DIGIT_FEATURES if c in orig.columns]

print(len(DIGIT_FEATURES), "DIGIT Features Created!")
print(DIGIT_FEATURES[:30])


Feature Engineering: DIGIT
67 DIGIT Features Created!
['Year_sign', 'Year_digit_1s', 'Year_digit_10s', 'Year_digit_100s', 'Year_digit_1000s', 'PitStop_sign', 'PitStop_digit_1s', 'PitStop_digit_10s', 'PitStop_digit_100s', 'PitStop_digit_1000s', 'LapNumber_sign', 'LapNumber_digit_1s', 'LapNumber_digit_10s', 'LapNumber_digit_100s', 'LapNumber_digit_1000s', 'Stint_sign', 'Stint_digit_1s', 'Stint_digit_10s', 'Stint_digit_100s', 'Stint_digit_1000s', 'TyreLife_sign', 'TyreLife_digit_1s', 'TyreLife_digit_10s', 'TyreLife_digit_100s', 'TyreLife_digit_1000s', 'Position_sign', 'Position_digit_1s', 'Position_digit_10s', 'Position_digit_100s', 'Position_digit_1000s']


In [9]:
# ============================================================
# Feature Engineering: shared001v2-style features for 041
# ============================================================

print_section("Feature Engineering: shared001v2-style features for 041")

SHARED001V2_FLOOR_CAT_COLS_041 = [
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
]

SHARED001V2_BIN_COLS_041 = [
    "RaceProgress_200_quantile_bin_",
    "LapTime (s)_7_quantile_bin_",
]

SHARED001V2_CROSS_COLS_041 = [
    "Race_Compound_",
    "Race_Year_",
]

SHARED001V2_COUNT_SOURCE_COLS_041 = [
    "PitStop_cat_",
    "RaceProgress_200_quantile_bin_",
    "LapTime (s)_7_quantile_bin_",
    "Race_Compound_",
    "Race_Year_",
    "Compound",
    "Race",
    "Driver",
]

SHARED001V2_CAT_FEATURES_041 = []
SHARED001V2_COUNT_FEATURES_041 = []


def _safe_qbin_041(s: pd.Series, q: int, name: str) -> pd.Series:
    """Robust target-free quantile binning fitted on the combined unsupervised table."""
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().sum() == 0:
        return pd.Series([f"{name}_missing"] * len(s), index=s.index, dtype="object")

    r = x.rank(method="first")
    try:
        b = pd.qcut(r, q=q, labels=False, duplicates="drop")
    except Exception:
        b = pd.cut(r, bins=min(q, max(int(r.nunique()), 1)), labels=False)

    return (
        pd.Series(b, index=s.index)
        .astype("Int64")
        .astype(str)
        .replace("<NA>", f"{name}_missing")
        .map(lambda z: f"{name}_{z}")
        .astype(str)
    )


def _build_shared001v2_features_041(all_df: pd.DataFrame) -> pd.DataFrame:
    """Add 029/038-style target-free representation features.

    This block intentionally does not add the 039 LOG features.
    """
    out = all_df.copy()

    if not CFG.USE_SHARED001V2_FEATURES_041:
        return out

    if "PitStop" in out.columns:
        pit = pd.to_numeric(out["PitStop"], errors="coerce").fillna(-1)
        out["PitStop_cat_"] = np.floor(pit).astype(np.int16).astype(str)

    for col in SHARED001V2_FLOOR_CAT_COLS_041:
        if col not in out.columns:
            continue

        x = pd.to_numeric(out[col], errors="coerce")
        cat_col = f"{col}_cat_"
        out[cat_col] = (
            np.floor(x.fillna(-9999))
            .astype(np.int32)
            .astype(str)
        )

        if cat_col not in SHARED001V2_COUNT_SOURCE_COLS_041:
            SHARED001V2_COUNT_SOURCE_COLS_041.append(cat_col)

    if "RaceProgress" in out.columns:
        out["RaceProgress_200_quantile_bin_"] = _safe_qbin_041(
            out["RaceProgress"], q=200, name="RP200"
        )

    if "LapTime (s)" in out.columns:
        out["LapTime (s)_7_quantile_bin_"] = _safe_qbin_041(
            out["LapTime (s)"], q=7, name="LT7"
        )

    if CFG.ADD_RACE_COMPOUND_YEAR_CROSSES_041:
        if {"Race", "Compound"}.issubset(out.columns):
            out["Race_Compound_"] = (
                out["Race"].astype(str) + "__" + out["Compound"].astype(str)
            )

        if {"Race", "Year"}.issubset(out.columns):
            out["Race_Year_"] = (
                out["Race"].astype(str) + "__" + out["Year"].astype(str)
            )

    count_cols = [
        c for c in SHARED001V2_COUNT_SOURCE_COLS_041
        if c in out.columns
    ]

    for col in count_cols:
        vc = out[col].astype(str).value_counts(dropna=False)

        if col == "PitStop_cat_":
            count_name = "_PitStop_cat_count"
        else:
            count_name = f"{col}_count_041"

        out[count_name] = (
            out[col].astype(str).map(vc).fillna(0).astype(np.float32)
        )

    return out


if CFG.USE_SHARED001V2_FEATURES_041:
    # Fit the target-free bin/count vocabulary on the combined train/test/original table.
    # This follows the same transductive availability as frequency/count encoding.
    n_train = len(train)
    n_test = len(test)
    n_orig = len(orig) if orig is not None else 0

    train_feat = train.drop(columns=[TARGET], errors="ignore").copy()
    test_feat = test.copy()

    frames = [train_feat, test_feat]

    if orig is not None:
        orig_feat = orig.drop(columns=[TARGET], errors="ignore").copy()
        frames.append(orig_feat)

    combined_feat = pd.concat(frames, axis=0, ignore_index=True, sort=False)
    combined_feat = _build_shared001v2_features_041(combined_feat)

    train_added = combined_feat.iloc[:n_train].reset_index(drop=True)
    test_added = combined_feat.iloc[n_train:n_train + n_test].reset_index(drop=True)

    for col in train_added.columns:
        if col not in train.columns:
            train[col] = train_added[col].values
        if col not in test.columns:
            test[col] = test_added[col].values

    if orig is not None:
        orig_added = combined_feat.iloc[n_train + n_test:n_train + n_test + n_orig].reset_index(drop=True)
        for col in orig_added.columns:
            if col not in orig.columns:
                orig[col] = orig_added[col].values

    for col in [
        "PitStop_cat_",
        "RaceProgress_200_quantile_bin_",
        "LapTime (s)_7_quantile_bin_",
        "Race_Compound_",
        "Race_Year_",
    ]:
        if col in train.columns and col in test.columns:
            SHARED001V2_CAT_FEATURES_041.append(col)

    for col in SHARED001V2_FLOOR_CAT_COLS_041:
        cat_col = f"{col}_cat_"
        if cat_col in train.columns and cat_col in test.columns:
            SHARED001V2_CAT_FEATURES_041.append(cat_col)

    SHARED001V2_CAT_FEATURES_041 = list(dict.fromkeys(SHARED001V2_CAT_FEATURES_041))

    for col in train.columns:
        if col == "_PitStop_cat_count" or col.endswith("_count_041"):
            if col in test.columns:
                SHARED001V2_COUNT_FEATURES_041.append(col)

    SHARED001V2_COUNT_FEATURES_041 = list(dict.fromkeys(SHARED001V2_COUNT_FEATURES_041))

    # Treat the added categorical/bin views like 016's NUM_as_CAT block:
    # they receive single-column frequency encoding and single-column TE.
    for col in SHARED001V2_CAT_FEATURES_041:
        if col not in NUM_as_CAT:
            NUM_as_CAT.append(col)

    # XGBoost categorical support requires category dtype for non-TE object columns.
    for df in [train, test] + ([orig] if orig is not None else []):
        for col in SHARED001V2_CAT_FEATURES_041:
            if col in df.columns:
                df[col] = df[col].astype(str).astype("category")

print("shared001v2 cat features 041:", len(SHARED001V2_CAT_FEATURES_041))
print(SHARED001V2_CAT_FEATURES_041)
print("shared001v2 count features 041:", len(SHARED001V2_COUNT_FEATURES_041))
print(SHARED001V2_COUNT_FEATURES_041)

assert not CFG.ADD_039_LOG_FEATURES, "041 must not add 039 LOG features."



Feature Engineering: shared001v2-style features for 041
shared001v2 cat features 041: 14
['PitStop_cat_', 'RaceProgress_200_quantile_bin_', 'LapTime (s)_7_quantile_bin_', 'Race_Compound_', 'Race_Year_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_']
shared001v2 count features 041: 17
['_PitStop_cat_count', 'RaceProgress_200_quantile_bin__count_041', 'LapTime (s)_7_quantile_bin__count_041', 'Race_Compound__count_041', 'Race_Year__count_041', 'Compound_count_041', 'Race_count_041', 'Driver_count_041', 'LapNumber_cat__count_041', 'Stint_cat__count_041', 'TyreLife_cat__count_041', 'Position_cat__count_041', 'LapTime (s)_cat__count_041', 'LapTime_Delta_cat__count_041', 'Cumulative_Degradation_cat__count_041', 'RaceProgress_cat__count_041', 'Position_Change_cat__count_041']


In [10]:
# ============================================================
# Model Feature Configuration
# ============================================================

print_section("Model Feature Configuration")

TE_BASE = [
    "Driver",
    "Compound",
    "Race",
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "RaceProgress",
    "Position_Change",
]

TE_BASE = [c for c in TE_BASE if c in train.columns and c in test.columns]
if orig is not None:
    TE_BASE = [c for c in TE_BASE if c in orig.columns]

BIGRAM_SPECS = list(combinations(TE_BASE, 2))

FEATURES = BASE + NUM_as_CAT + DIGIT_FEATURES + SHARED001V2_COUNT_FEATURES_041
FEATURES = [c for c in FEATURES if c in train.columns and c in test.columns]
if orig is not None:
    FEATURES = [c for c in FEATURES if c in orig.columns]

# Deduplicate while preserving order
FEATURES = list(dict.fromkeys(FEATURES))

print(len(BIGRAM_SPECS), "BIGRAM specs")
print(len(FEATURES), "Base features")
print(len(SHARED001V2_CAT_FEATURES_041), "shared001v2 cat features 041")
print(len(SHARED001V2_COUNT_FEATURES_041), "shared001v2 count features 041")
print("FEATURES sample:", FEATURES[:30])

assert CFG.DANGER_COL not in FEATURES


Model Feature Configuration
55 BIGRAM specs
135 Base features
14 shared001v2 cat features 041
17 shared001v2 count features 041
FEATURES sample: ['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'LapTime (s)_cat', 'LapTime_Delta_cat', 'Cumulative_Degradation_cat', 'LapTime (s)_round1_cat', 'LapTime (s)_round0_cat', 'LapTime (s)_round_step_0p5_cat', 'LapTime (s)_round_step_1p0_cat', 'LapTime (s)_round_step_2p0_cat', 'LapTime (s)_round_step_5p0_cat', 'LapTime_Delta_round1_cat', 'LapTime_Delta_round0_cat', 'LapTime_Delta_round_step_0p5_cat', 'LapTime_Delta_round_step_1p0_cat', 'LapTime_Delta_round_step_2p0_cat', 'LapTime_Delta_round_step_5p0_cat', 'LapTime_Delta_round_step_10p0_cat']


In [11]:
# ============================================================
# Encoding Helpers
# ============================================================

def make_inner_fold_ids(y, n_splits=5, seed=42):
    fold_ids = np.zeros(len(y), dtype=np.int32)

    inner = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )

    dummy_X = np.zeros((len(y), 1))

    for fold, (_, va_idx) in enumerate(inner.split(dummy_X, y)):
        fold_ids[va_idx] = fold

    return cudf.Series(fold_ids)


def add_frequency_encode(
    X_tr,
    X_va,
    X_test,
    cols,
    out_col=None,
    normalize=True,
):
    cols = list(cols)

    if out_col is None:
        out_col = "fe__" + "__".join(cols)

    X_tr_key = X_tr[cols].astype(str)
    X_va_key = X_va[cols].astype(str)
    X_test_key = X_test[cols].astype(str)

    denom = len(X_tr) if normalize else 1.0

    if len(cols) == 1:
        c = cols[0]
        freq_map = X_tr_key[c].value_counts(dropna=False)

        X_tr[out_col] = (
            X_tr_key[c]
            .map(freq_map)
            .fillna(0)
            .astype(np.float32)
            .values / denom
        )

        X_va[out_col] = (
            X_va_key[c]
            .map(freq_map)
            .fillna(0)
            .astype(np.float32)
            .values / denom
        )

        X_test[out_col] = (
            X_test_key[c]
            .map(freq_map)
            .fillna(0)
            .astype(np.float32)
            .values / denom
        )

    else:
        freq_df = (
            X_tr_key
            .groupby(cols, dropna=False)
            .size()
            .reset_index(name=out_col)
        )

        def map_joint_freq(df_key):
            return (
                df_key
                .reset_index(drop=True)
                .merge(freq_df, on=cols, how="left")[out_col]
                .fillna(0)
                .astype(np.float32)
                .values / denom
            )

        X_tr[out_col] = map_joint_freq(X_tr_key)
        X_va[out_col] = map_joint_freq(X_va_key)
        X_test[out_col] = map_joint_freq(X_test_key)


def add_cuml_te(
    X_tr,
    X_va,
    X_test,
    X_tr_g,
    X_va_g,
    X_test_g,
    y_tr_g,
    cols,
    out_col,
    fold_ids_g,
    seed=42,
    smooth=20,
    n_folds=5,
):
    cols = list(cols)

    te = cuTargetEncoder(
        n_folds=n_folds,
        smooth=smooth,
        seed=seed,
        split_method="random",
        output_type="numpy",
        stat="mean",
        multi_feature_mode="combination",
    )

    tr_enc = te.fit_transform(
        X_tr_g[cols],
        y_tr_g,
        fold_ids=fold_ids_g,
    )

    va_enc = te.transform(X_va_g[cols])
    test_enc = te.transform(X_test_g[cols])

    X_tr[out_col] = to_numpy(tr_enc).reshape(-1).astype(np.float32)
    X_va[out_col] = to_numpy(va_enc).reshape(-1).astype(np.float32)
    X_test[out_col] = to_numpy(test_enc).reshape(-1).astype(np.float32)

In [12]:
# ============================================================
# 5fold OOF Training
# ============================================================

print_section("5fold OOF Training")

oof_preds = np.zeros(len(train), dtype=np.float32)
test_preds = np.zeros(len(test), dtype=np.float32)

fold_records = []
seed_records = []
all_feature_cols_after_fe_te = None

skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

if orig is not None and CFG.USE_ORIGINAL:
    X_orig = orig[FEATURES].copy()
    y_orig = orig[TARGET].copy().reset_index(drop=True)
else:
    X_orig = None
    y_orig = None

for fold, (tr_idx, va_idx) in enumerate(skf.split(train[FEATURES], train[TARGET]), 1):
    print("\n" + "=" * 90)
    print(f"Fold {fold}/{CFG.N_SPLITS}")
    print("=" * 90)

    X_tr_base = train[FEATURES].iloc[tr_idx].copy().reset_index(drop=True)
    y_tr_base = train[TARGET].iloc[tr_idx].copy().reset_index(drop=True)

    X_va = train[FEATURES].iloc[va_idx].copy().reset_index(drop=True)
    y_va = train[TARGET].iloc[va_idx].copy().reset_index(drop=True)

    X_test = test[FEATURES].copy().reset_index(drop=True)

    if X_orig is not None and CFG.USE_ORIGINAL:
        X_tr = pd.concat(
            [X_tr_base.reset_index(drop=True), X_orig.reset_index(drop=True)],
            axis=0,
            ignore_index=True,
        )
        y_tr = pd.concat(
            [y_tr_base.reset_index(drop=True), y_orig.reset_index(drop=True)],
            axis=0,
            ignore_index=True,
        )
    else:
        X_tr = X_tr_base.copy()
        y_tr = y_tr_base.copy()

    print("Base:", X_tr.shape, X_va.shape, X_test.shape)
    print("target mean train fold:", float(y_tr.mean()))
    print("target mean valid fold:", float(y_va.mean()))

    te_source_cols = sorted(set(["Driver"] + NUM_as_CAT + TE_BASE))
    te_source_cols = [c for c in te_source_cols if c in X_tr.columns]

    X_tr_g = cudf.from_pandas(X_tr[te_source_cols].astype(str))
    X_va_g = cudf.from_pandas(X_va[te_source_cols].astype(str))
    X_test_g = cudf.from_pandas(X_test[te_source_cols].astype(str))
    y_tr_g = cudf.Series(y_tr.values)

    te_seed = 1000 + fold

    fold_ids_g = make_inner_fold_ids(
        y_tr,
        n_splits=CFG.N_TE_FOLDS,
        seed=te_seed,
    )

    # -------------------------
    # Frequency Encoding
    # -------------------------

    FE_SINGLE_COLS = sorted(set(["Driver"] + NUM_as_CAT))
    FE_SINGLE_COLS = [c for c in FE_SINGLE_COLS if c in X_tr.columns]

    for c in FE_SINGLE_COLS:
        add_frequency_encode(
            X_tr,
            X_va,
            X_test,
            cols=(c,),
            out_col=f"fe__{c}",
            normalize=True,
        )

    for cols in BIGRAM_SPECS:
        if all(c in X_tr.columns for c in cols):
            add_frequency_encode(
                X_tr,
                X_va,
                X_test,
                cols=cols,
                out_col="fe2__" + "__".join(cols),
                normalize=True,
            )

    print("After FE:", X_tr.shape, X_va.shape, X_test.shape)

    # -------------------------
    # Target Encoding
    # -------------------------

    if "Driver" in X_tr.columns:
        add_cuml_te(
            X_tr,
            X_va,
            X_test,
            X_tr_g,
            X_va_g,
            X_test_g,
            y_tr_g,
            cols=("Driver",),
            out_col="te_Driver",
            fold_ids_g=fold_ids_g,
            seed=te_seed,
            smooth=CFG.TE_SMOOTH,
            n_folds=CFG.N_TE_FOLDS,
        )

    for c in NUM_as_CAT:
        if c in X_tr.columns:
            # shared_005 overwrote the original categorical string column with TE numeric.
            # We keep that behavior for comparability.
            add_cuml_te(
                X_tr,
                X_va,
                X_test,
                X_tr_g,
                X_va_g,
                X_test_g,
                y_tr_g,
                cols=(c,),
                out_col=c,
                fold_ids_g=fold_ids_g,
                seed=te_seed,
                smooth=CFG.TE_SMOOTH,
                n_folds=CFG.N_TE_FOLDS,
            )

    for cols in BIGRAM_SPECS:
        if all(c in X_tr.columns for c in cols):
            add_cuml_te(
                X_tr,
                X_va,
                X_test,
                X_tr_g,
                X_va_g,
                X_test_g,
                y_tr_g,
                cols=cols,
                out_col="te2__" + "__".join(cols),
                fold_ids_g=fold_ids_g,
                seed=te_seed,
                smooth=CFG.TE_SMOOTH,
                n_folds=CFG.N_TE_FOLDS,
            )

    print("After TE:", X_tr.shape, X_va.shape, X_test.shape)

    if all_feature_cols_after_fe_te is None:
        all_feature_cols_after_fe_te = list(X_tr.columns)

    fold_va_preds = np.zeros(len(X_va), dtype=np.float32)
    fold_test_preds = np.zeros(len(X_test), dtype=np.float32)

    seed_aucs = []
    seed_best_iters = []

    for s, seed in enumerate(CFG.SEEDS):
        model_seed = seed + (fold - 1) * 100

        print(f"  Seed {s + 1}/{len(CFG.SEEDS)}: {model_seed}")

        params = CFG.XGB_PARAMS.copy()
        params["random_state"] = model_seed

        model = XGBClassifier(**params)

        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            verbose=CFG.VERBOSE,
        )

        va_pred = clip_pred(model.predict_proba(X_va)[:, 1]).astype(np.float32)
        test_pred = clip_pred(model.predict_proba(X_test)[:, 1]).astype(np.float32)

        fold_va_preds += va_pred / len(CFG.SEEDS)
        fold_test_preds += test_pred / len(CFG.SEEDS)

        seed_auc = roc_auc_score(y_va, va_pred)

        try:
            best_iter = int(model.best_iteration)
        except Exception:
            best_iter = -1

        seed_aucs.append(seed_auc)
        seed_best_iters.append(best_iter)

        seed_records.append({
            "fold": fold,
            "seed_index": s + 1,
            "model_seed": model_seed,
            "auc": float(seed_auc),
            "best_iteration": int(best_iter),
            "n_train": int(len(X_tr)),
            "n_valid": int(len(X_va)),
            "n_features": int(X_tr.shape[1]),
        })

        print(f"    Seed AUC: {seed_auc:.9f}")
        print(f"    Best iteration: {best_iter}")

        del model, va_pred, test_pred
        gc.collect()

    oof_preds[va_idx] = fold_va_preds
    test_preds += fold_test_preds / CFG.N_SPLITS

    fold_auc = roc_auc_score(y_va, fold_va_preds)
    fold_logloss = log_loss(y_va, clip_pred(fold_va_preds))

    print(f"Fold {fold} Ensemble AUC    : {fold_auc:.9f}")
    print(f"Fold {fold} Ensemble LogLoss: {fold_logloss:.9f}")

    fold_records.append({
        "fold": fold,
        "auc": float(fold_auc),
        "logloss": float(fold_logloss),
        "seed_auc_mean": float(np.mean(seed_aucs)),
        "seed_auc_std": float(np.std(seed_aucs)),
        "seed_best_iter_mean": float(np.mean(seed_best_iters)),
        "seed_best_iter_min": int(np.min(seed_best_iters)),
        "seed_best_iter_max": int(np.max(seed_best_iters)),
        "n_train_comp": int(len(X_tr_base)),
        "n_train_orig": int(len(X_orig)) if X_orig is not None and CFG.USE_ORIGINAL else 0,
        "n_valid": int(len(X_va)),
        "n_features_after_fe_te": int(X_tr.shape[1]),
    })

    del X_tr_base, y_tr_base, X_va, y_va, X_test, X_tr, y_tr
    del X_tr_g, X_va_g, X_test_g, y_tr_g, fold_ids_g
    gc.collect()


5fold OOF Training

Fold 1/5
Base: (452683, 135) (87828, 135) (188165, 135)
target mean train fold: 0.21147911452385001
target mean valid fold: 0.19899121009245344
After FE: (452683, 228) (87828, 228) (188165, 228)
After TE: (452683, 284) (87828, 284) (188165, 284)
  Seed 1/5: 42
[0]	validation_0-auc:0.92706
[200]	validation_0-auc:0.94801
[400]	validation_0-auc:0.95053
[600]	validation_0-auc:0.95146
[800]	validation_0-auc:0.95193
[1000]	validation_0-auc:0.95218
[1200]	validation_0-auc:0.95238
[1400]	validation_0-auc:0.95252
[1600]	validation_0-auc:0.95260
[1800]	validation_0-auc:0.95268
[2000]	validation_0-auc:0.95274
[2200]	validation_0-auc:0.95277
[2400]	validation_0-auc:0.95276
[2414]	validation_0-auc:0.95276
    Seed AUC: 0.952783820
    Best iteration: 2214
  Seed 2/5: 43
[0]	validation_0-auc:0.92820
[200]	validation_0-auc:0.94786
[400]	validation_0-auc:0.95053
[600]	validation_0-auc:0.95137
[800]	validation_0-auc:0.95177
[1000]	validation_0-auc:0.95208
[1200]	validation_0-auc:0.

In [13]:
# ============================================================
# CV Result
# ============================================================

print_section("CV Result")

cv_auc = roc_auc_score(train[TARGET], oof_preds)
cv_logloss = log_loss(train[TARGET], clip_pred(oof_preds))

fold_df = pd.DataFrame(fold_records)
seed_df = pd.DataFrame(seed_records)

print(f"Overall CV AUC    : {cv_auc:.9f}")
print(f"Overall CV LogLoss: {cv_logloss:.9f}")
print("fold auc mean:", fold_df["auc"].mean())
print("fold auc std :", fold_df["auc"].std())

display(fold_df)
display(seed_df.head())


CV Result
Overall CV AUC    : 0.952012441
Overall CV LogLoss: 0.219506622
fold auc mean: 0.9520186984319444
fold auc std : 0.0009007829707159916


,fold,auc,logloss,seed_auc_mean,seed_auc_std,seed_best_iter_mean,seed_best_iter_min,seed_best_iter_max,n_train_comp,n_train_orig,n_valid,n_features_after_fe_te
0,1,0.953201,0.216860,0.952722,0.000046,2373.8,2210,2683,351312,101371,87828,284
1,2,0.950964,0.221944,0.950518,0.000076,2076.6,1835,2271,351312,101371,87828,284
2,3,0.951748,0.219782,0.951301,0.000133,2060.4,1452,2718,351312,101371,87828,284
3,4,0.951520,0.220853,0.951025,0.000022,2675.6,2128,3422,351312,101371,87828,284
4,5,0.952660,0.218093,0.952190,0.000027,2390.2,1878,2572,351312,101371,87828,284


,fold,seed_index,model_seed,auc,best_iteration,n_train,n_valid,n_features
0,1,1,42,0.952784,2214,452683,87828,284
1,1,2,43,0.952717,2683,452683,87828,284
2,1,3,44,0.952762,2534,452683,87828,284
3,1,4,45,0.952675,2210,452683,87828,284
4,1,5,46,0.952670,2228,452683,87828,284


In [14]:
# ============================================================
# Save Artifacts
# ============================================================

print_section("Save Artifacts")

np.save(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy", oof_preds)
np.save(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy", test_preds)

oof_csv = pd.DataFrame({
    CFG.ID_COL: train_ids.values,
    TARGET: train[TARGET].values,
    "pred": oof_preds,
})
oof_csv.to_csv(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv", index=False)

sub = sample_submission.copy()
sub[TARGET] = clip_pred(test_preds)

sub_path = CFG.OUTDIR / f"submission_{CFG.EXP_ID}.csv"
sub.to_csv(sub_path, index=False)

fold_df.to_csv(CFG.OUTDIR / "fold_scores.csv", index=False)
seed_df.to_csv(CFG.OUTDIR / "seed_scores.csv", index=False)

# Feature columns after FE/TE
if all_feature_cols_after_fe_te is None:
    all_feature_cols_after_fe_te = FEATURES

feature_df = pd.DataFrame({
    "feature": all_feature_cols_after_fe_te,
    "is_base": [c in BASE for c in all_feature_cols_after_fe_te],
    "is_num_as_cat": [c in NUM_as_CAT for c in all_feature_cols_after_fe_te],
    "is_digit": [c in DIGIT_FEATURES for c in all_feature_cols_after_fe_te],
    "is_shared001v2_cat_041": [c in SHARED001V2_CAT_FEATURES_041 for c in all_feature_cols_after_fe_te],
    "is_shared001v2_count_041": [c in SHARED001V2_COUNT_FEATURES_041 for c in all_feature_cols_after_fe_te],
    "is_frequency": [c.startswith("fe__") or c.startswith("fe2__") for c in all_feature_cols_after_fe_te],
    "is_target_encoding": [c.startswith("te_") or c.startswith("te2__") for c in all_feature_cols_after_fe_te],
    "contains_year": ["Year" in c for c in all_feature_cols_after_fe_te],
    "contains_race": ["Race" in c for c in all_feature_cols_after_fe_te],
    "contains_driver": ["Driver" in c for c in all_feature_cols_after_fe_te],
    "contains_pitstop": ["PitStop" in c for c in all_feature_cols_after_fe_te],
})
feature_df.to_csv(CFG.OUTDIR / "feature_columns.csv", index=False)

pd.Series(FEATURES, name="base_feature").to_csv(CFG.OUTDIR / "base_features.csv", index=False)
pd.Series(NUM_as_CAT, name="num_as_cat_feature").to_csv(CFG.OUTDIR / "num_as_cat_features.csv", index=False)
pd.Series(DIGIT_FEATURES, name="digit_feature").to_csv(CFG.OUTDIR / "digit_features.csv", index=False)

shared001v2_df = pd.DataFrame({
    "feature": SHARED001V2_CAT_FEATURES_041 + SHARED001V2_COUNT_FEATURES_041,
    "kind": ["cat_or_bin"] * len(SHARED001V2_CAT_FEATURES_041) + ["count"] * len(SHARED001V2_COUNT_FEATURES_041),
})
shared001v2_df.to_csv(CFG.OUTDIR / "shared001v2_features_041.csv", index=False)

bigram_df = pd.DataFrame({
    "col1": [a for a, b in BIGRAM_SPECS],
    "col2": [b for a, b in BIGRAM_SPECS],
    "fe_feature": ["fe2__" + "__".join((a, b)) for a, b in BIGRAM_SPECS],
    "te_feature": ["te2__" + "__".join((a, b)) for a, b in BIGRAM_SPECS],
})
bigram_df.to_csv(CFG.OUTDIR / "bigram_specs.csv", index=False)


Save Artifacts


In [15]:
# ============================================================
# Save cv_result.json
# ============================================================

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
    },
    "source": {
        "shared_code": "shared_005 + 029/038 shared001v2 feature transfer",
        "conversion": "016 XGBoost FE/TE seed ensemble with 029/038-style feature representation transferred to XGB",
        "note": [
            "This is not shared_005 as-is submission.",
            "The implementation keeps 016 shared_005 feature design and 5fold x 5seed XGB ensemble.",
            "041 adds 029/038-style PitStop_cat_, floor cats, quantile bins, Race_Compound_/Race_Year_, and count encodings.",
            "039 LOG features are intentionally not added in this first XGB transfer test.",
            "Original rows are appended to each fold train side only.",
            "Competition validation rows only are used for OOF.",
            "OOF/pred npy files are saved for blend.",
            "Normalized_TyreLife is explicitly dropped from original.",
        ],
    },
    "data": {
        "train_shape": list(train.shape),
        "test_shape": list(test.shape),
        "original_available": orig is not None,
        "original_shape_after_drop": list(orig.shape) if orig is not None else None,
        "target": CFG.TARGET,
        "id_col": CFG.ID_COL,
        "danger_col": CFG.DANGER_COL,
        "danger_col_used": False,
        "use_original_rows": CFG.USE_ORIGINAL,
        "competition_target_mean": float(train[CFG.TARGET].mean()),
        "original_target_mean": float(orig[CFG.TARGET].mean()) if orig is not None else None,
    },
    "cv": {
        "scheme": "StratifiedKFold",
        "n_splits": CFG.N_SPLITS,
        "shuffle": True,
        "random_state": CFG.SEED,
        "target_encoding_inner_folds": CFG.N_TE_FOLDS,
        "target_encoding_smooth": CFG.TE_SMOOTH,
    },
    "features": {
        "base_feature_count_before_fe_te": int(len(FEATURES)),
        "final_feature_count_after_fe_te": int(len(all_feature_cols_after_fe_te)),
        "num_as_cat_count": int(len(NUM_as_CAT)),
        "digit_feature_count": int(len(DIGIT_FEATURES)),
        "bigram_spec_count": int(len(BIGRAM_SPECS)),
        "feature_blocks": [
            "original_baseline_features",
            "numeric_as_categorical_features",
            "digit_features",
            "single_column_frequency_encoding",
            "pairwise_joint_frequency_encoding",
            "single_column_target_encoding",
            "pairwise_target_encoding",
            "shared001v2_cat_bin_features_041",
            "shared001v2_count_features_041",
        ],
        "shared001v2_cat_feature_count_041": int(len(SHARED001V2_CAT_FEATURES_041)),
        "shared001v2_count_feature_count_041": int(len(SHARED001V2_COUNT_FEATURES_041)),
        "shared001v2_features_041": list(SHARED001V2_CAT_FEATURES_041 + SHARED001V2_COUNT_FEATURES_041),
        "risk_features_to_watch": [
            "Year",
            "Race",
            "Race-Year interactions via pairwise FE/TE",
            "PitStop",
            "LapNumber/TyreLife/RaceProgress pairwise FE/TE",
        ],
        "normalized_tyrelife_policy": [
            "Normalized_TyreLife is dropped.",
            "Direct use is forbidden.",
            "Intentional proxy reconstruction is not added.",
        ],
    },
    "model": {
        "family": "XGBoost",
        "estimator": "XGBClassifier",
        "outer_folds": CFG.N_SPLITS,
        "seeds_per_fold": len(CFG.SEEDS),
        "total_models": int(CFG.N_SPLITS * len(CFG.SEEDS)),
        "seeds": CFG.SEEDS,
        "params": CFG.XGB_PARAMS,
    },
    "result": {
        "cv_auc": float(cv_auc),
        "cv_logloss": float(cv_logloss),
        "fold_auc_mean": float(fold_df["auc"].mean()),
        "fold_auc_std": float(fold_df["auc"].std()),
        "fold_scores": fold_records,
        "seed_auc_mean": float(seed_df["auc"].mean()),
        "seed_auc_std": float(seed_df["auc"].std()),
    },
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "oof_npy": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.npy"),
        "pred_npy": str(CFG.OUTDIR / f"pred_{CFG.EXP_ID}.npy"),
        "oof_csv": str(CFG.OUTDIR / f"oof_{CFG.EXP_ID}.csv"),
        "submission": str(sub_path),
        "fold_scores": str(CFG.OUTDIR / "fold_scores.csv"),
        "seed_scores": str(CFG.OUTDIR / "seed_scores.csv"),
        "feature_columns": str(CFG.OUTDIR / "feature_columns.csv"),
        "base_features": str(CFG.OUTDIR / "base_features.csv"),
        "num_as_cat_features": str(CFG.OUTDIR / "num_as_cat_features.csv"),
        "digit_features": str(CFG.OUTDIR / "digit_features.csv"),
        "bigram_specs": str(CFG.OUTDIR / "bigram_specs.csv"),
        "shared001v2_features_041": str(CFG.OUTDIR / "shared001v2_features_041.csv"),
        "cv_result": str(CFG.OUTDIR / "cv_result.json"),
        "memo_draft": str(CFG.OUTDIR / "memo_draft.yml"),
    },
    "decision_policy": {
        "single_model": [
            "Do not adopt by single CV alone.",
            "Compare against 016, 017, and 040.",
            "016 CV was 0.9519855866 / LB 0.95164; 017 CV was 0.9519393683 / LB 0.95164; 040 CV was 0.9524028165 / LB 0.95195.",
        ],
        "blend": [
            "Add 041 to current best stack and compare add041 against add040.",
            "Evaluate avg / Ridge / ElasticNet / LogReg / HC / NM.",
            "If 041 receives natural positive weight and improves CV/LB, keep.",
            "If 041 is zero-weight across HC/NM/LogReg and does not improve avg, drop.",
        ],
    },
}

with open(CFG.OUTDIR / "cv_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

memo_yml = f"""experiment:
  id: {CFG.EXP_ID}
  competition: {CFG.COMPETITION}
  metric: {CFG.METRIC}
  status: completed_single_pending_public_lb_and_blend

objective:
  summary: >
    016 XGB shared005 FE/TE seed ensembleをベースに、
    029/038 RealMLP shared001v2系で効いている特徴表現をXGBoostへ移植する。
    目的は単体主力化ではなく、XGB branchの016/017を更新し、
    add041 blendでXGB補助素材として機能するか確認する。
  role: xgb_branch_update_candidate

base:
  experiment: exp_20260508_016_xgb_shared005_fe_te_seedens_min
  model: XGBClassifier
  family: XGBoost
  reference_scores:
    "016":
      cv_auc: 0.9519855866173828
      public_lb: 0.95164
    "017":
      cv_auc: 0.9519393682635370
      public_lb: 0.95164
    "040_lgb_reference":
      cv_auc: 0.9524028165133429
      public_lb: 0.95195

changes_from_016:
  add_shared001v2_features_041:
    - PitStop_cat_
    - _PitStop_cat_count
    - floored numerical category features
    - RaceProgress_200_quantile_bin_
    - LapTime (s)_7_quantile_bin_
    - Race_Compound_
    - Race_Year_
    - count encoding for added categorical/bin features
  intentionally_not_added:
    - 039 LOG features
    - pseudo label
    - Normalized_TyreLife
    - Normalized_TyreLife proxy
    - future endpoint proxy
    - Optuna

kept_from_016:
  model: XGBClassifier
  original_rows_appended_to_fold_train: {CFG.USE_ORIGINAL}
  outer_cv:
    scheme: StratifiedKFold
    n_splits: {CFG.N_SPLITS}
    shuffle: true
    random_state: {CFG.SEED}
  inner_te_folds: {CFG.N_TE_FOLDS}
  seeds_per_fold: {CFG.SEEDS}
  xgb_params:
    n_estimators: {CFG.XGB_PARAMS.get("n_estimators")}
    learning_rate: {CFG.XGB_PARAMS.get("learning_rate")}
    grow_policy: {CFG.XGB_PARAMS.get("grow_policy")}
    max_depth: {CFG.XGB_PARAMS.get("max_depth")}
    max_leaves: {CFG.XGB_PARAMS.get("max_leaves")}
    min_child_weight: {CFG.XGB_PARAMS.get("min_child_weight")}
    subsample: {CFG.XGB_PARAMS.get("subsample")}
    colsample_bytree: {CFG.XGB_PARAMS.get("colsample_bytree")}
    reg_alpha: {CFG.XGB_PARAMS.get("reg_alpha")}
    reg_lambda: {CFG.XGB_PARAMS.get("reg_lambda")}
    tree_method: {CFG.XGB_PARAMS.get("tree_method")}
    device: {CFG.XGB_PARAMS.get("device")}
    enable_categorical: {CFG.XGB_PARAMS.get("enable_categorical")}

features:
  base_feature_count_before_fe_te: {len(FEATURES)}
  final_feature_count_after_fe_te: {len(all_feature_cols_after_fe_te)}
  num_as_cat_count: {len(NUM_as_CAT)}
  digit_feature_count: {len(DIGIT_FEATURES)}
  bigram_spec_count: {len(BIGRAM_SPECS)}
  shared001v2_cat_feature_count_041: {len(SHARED001V2_CAT_FEATURES_041)}
  shared001v2_count_feature_count_041: {len(SHARED001V2_COUNT_FEATURES_041)}

result:
  cv_auc: {cv_auc:.12f}
  cv_logloss: {cv_logloss:.12f}
  fold_auc_mean: {fold_df["auc"].mean():.12f}
  fold_auc_std: {fold_df["auc"].std():.12f}
  public_lb: null

comparison:
  vs_016:
    cv_016: 0.9519855866173828
    public_lb_016: 0.95164
    delta_cv: "{cv_auc - 0.9519855866173828:+.12f}"
  vs_017:
    cv_017: 0.9519393682635370
    public_lb_017: 0.95164
    delta_cv: "{cv_auc - 0.9519393682635370:+.12f}"
  vs_040_lgb_reference:
    cv_040: 0.9524028165133429
    public_lb_040: 0.95195
    delta_cv: "{cv_auc - 0.9524028165133429:+.12f}"

artifacts:
  outdir: {str(CFG.OUTDIR)}
  oof_npy: oof_{CFG.EXP_ID}.npy
  pred_npy: pred_{CFG.EXP_ID}.npy
  submission: submission_{CFG.EXP_ID}.csv
  fold_scores: fold_scores.csv
  seed_scores: seed_scores.csv
  feature_columns: feature_columns.csv
  shared001v2_features_041: shared001v2_features_041.csv
  cv_result: cv_result.json
  memo_draft: memo_draft.yml

blend_policy:
  benchmark_notebook: ps-s6e5-033-blend-logreg-topk-feature-variants-min-Add041.ipynb
  compare_methods:
    - avg
    - hc_nonnegative_raw
    - nm_softmax_raw
    - ridge_meta_pruned
    - logreg_meta_pruned
  decision_rule:
    keep:
      - single_cv_above_016_017
      - public_lb_above_016_017
      - add041_nm_or_logreg_improves_add040
      - useful_weight_in_nm_or_hc
    hold:
      - single_cv_above_016_017_but_no_blend_update
      - low_corr_against_realmlp_core_but_weak_single
    drop:
      - single_cv_not_above_016_017
      - public_lb_not_above_016_017
      - no_weight_in_blend

judgement:
  status: pending_public_lb_and_blend
  summary: >
    041はXGB branchの更新確認であり、単体RealMLP級を期待する実験ではない。
    016/017より明確に強く、blendでXGB branchの置換または併用価値が出るかで判断する。
"""
with open(CFG.OUTDIR / "memo_draft.yml", "w", encoding="utf-8") as f:
    f.write(memo_yml)


print("Saved to:", CFG.OUTDIR)
print("Submission:", sub_path)
print(json.dumps(result, ensure_ascii=False, indent=2))

Saved to: /kaggle/working/exp_20260520_041_xgb_shared001v2_features_min
Submission: /kaggle/working/exp_20260520_041_xgb_shared001v2_features_min/submission_exp_20260520_041_xgb_shared001v2_features_min.csv
{
  "experiment": {
    "id": "exp_20260520_041_xgb_shared001v2_features_min",
    "competition": "PS S6E5 Predicting F1 Pit Stops",
    "metric": "AUC"
  },
  "source": {
    "shared_code": "shared_005 + 029/038 shared001v2 feature transfer",
    "conversion": "016 XGBoost FE/TE seed ensemble with 029/038-style feature representation transferred to XGB",
    "note": [
      "This is not shared_005 as-is submission.",
      "The implementation keeps 016 shared_005 feature design and 5fold x 5seed XGB ensemble.",
      "041 adds 029/038-style PitStop_cat_, floor cats, quantile bins, Race_Compound_/Race_Year_, and count encodings.",
      "039 LOG features are intentionally not added in this first XGB transfer test.",
      "Original rows are appended to each fold train side only.",
 

In [16]:
# ============================================================
# Preview
# ============================================================

print_section("OOF Preview")
display(oof_csv.head())

print_section("Submission Preview")
display(sub.head())

print_section("Feature Columns Preview")
display(feature_df.head(30))
print_section("shared001v2 Features 041 Preview")
display(shared001v2_df.head(50))



OOF Preview


,id,PitNextLap,pred
0,0,1.0,0.812299
1,1,0.0,0.477731
2,2,1.0,0.456612
3,3,0.0,0.001398
4,4,0.0,0.187443



Submission Preview


,id,PitNextLap
0,439140,0.004075
1,439141,0.011856
2,439142,0.005124
3,439143,0.379919
4,439144,0.811519



Feature Columns Preview


,feature,is_base,is_num_as_cat,is_digit,is_shared001v2_cat_041,is_shared001v2_count_041,is_frequency,is_target_encoding,contains_year,contains_race,contains_driver,contains_pitstop
0,Driver,True,False,False,False,False,False,False,False,False,True,False
1,Compound,True,False,False,False,False,False,False,False,False,False,False
2,Race,True,False,False,False,False,False,False,False,True,False,False
3,Year,True,False,False,False,False,False,False,True,False,False,False
4,PitStop,True,False,False,False,False,False,False,False,False,False,True
5,LapNumber,True,False,False,False,False,False,False,False,False,False,False
6,Stint,True,False,False,False,False,False,False,False,False,False,False
7,TyreLife,True,False,False,False,False,False,False,False,False,False,False
8,Position,True,False,False,False,False,False,False,False,False,False,False
9,LapTime (s),True,False,False,False,False,False,False,False,False,False,False



shared001v2 Features 041 Preview


,feature,kind
0,PitStop_cat_,cat_or_bin
1,RaceProgress_200_quantile_bin_,cat_or_bin
2,LapTime (s)_7_quantile_bin_,cat_or_bin
3,Race_Compound_,cat_or_bin
4,Race_Year_,cat_or_bin
5,LapNumber_cat_,cat_or_bin
6,Stint_cat_,cat_or_bin
7,TyreLife_cat_,cat_or_bin
8,Position_cat_,cat_or_bin
9,LapTime (s)_cat_,cat_or_bin
